# 15. 장르 × 가격대 조합별 유저 반응 확보율 × 초기 만족도 2x2 분석

**분석 목적:** 단일 장르나 단일 가격대만으로는 설명되지 않는 출시 전략 차이를 보기 위해, `장르 × 가격대` 조합별로 `유저 반응 확보율`과 `초기 만족도 high 비율`을 함께 비교한다.

**핵심 질문:** 어떤 장르·가격대 조합은 유저의 첫 반응은 잘 얻지만 만족도로 이어지지 않고, 어떤 조합은 반응도 얻고 만족도도 높은가?

**해석 프레임:**
- x축: `유저 반응 확보율(%)` = 해당 장르·가격대 조합 게임 중 리뷰 `10~49개`에 도달한 비율
- y축: `초기 만족도 high 비율(%)` = 유저 반응을 확보한 게임 중 긍정률 `80% 이상` 비율
- 우상단: 반응 확보와 만족도가 모두 강한 조합
- 우하단: 반응은 얻지만 만족도가 약한 조합
- 좌상단: 반응은 적지만 반응 후 만족도는 높은 조합
- 좌하단: 반응 확보와 만족도 모두 약한 조합

In [12]:
import ast
from pathlib import Path

import pandas as pd
import plotly.express as px
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import fdrcorrection


## 1. 데이터 로드 및 분석 기준 정의

침묵 그룹(`리뷰 0~9개`)과 유저 반응 그룹(`리뷰 10~49개`)을 결합한 뒤, `장르 × 가격대` 조합별 반응 확보율과 만족도 비율을 계산한다.

In [13]:
DATA_DIR = Path('../../../data/preprocessed')
SILENCE_PATH = DATA_DIR / 'steam_indie_games_silence.csv'
GRADED_PATH = DATA_DIR / 'steam_indie_games_graded.csv'

silence = pd.read_csv(SILENCE_PATH)
response_all = pd.read_csv(GRADED_PATH)
response = response_all[response_all['scale_grade'] == 'low'].copy()

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']
PRICE_BINS = [0, 5, 10, 15, 20, 30, 60, float('inf')]
PRICE_LABELS = ['~$5', '$5~10', '$10~15', '$15~20', '$20~30', '$30~60', '$60+']
MIN_TOTAL_GAMES = 30
MIN_RESPONSE_GAMES = 10


def parse_list_column(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

for df in [silence, response]:
    df['genres'] = df['genres'].apply(parse_list_column)
    df['price_range'] = pd.cut(df['price'], bins=PRICE_BINS, labels=PRICE_LABELS, right=False, include_lowest=True)

response['is_high_satisfaction'] = response['satisfaction_grade'].eq('high')

overall_response_capture_rate = len(response) / (len(response) + len(silence)) * 100
overall_high_satisfaction_rate = response['is_high_satisfaction'].mean() * 100

print(f'침묵 그룹: {len(silence):,}개')
print(f'유저 반응 그룹: {len(response):,}개')
print(f'전체 평균 유저 반응 확보율: {overall_response_capture_rate:.1f}%')
print(f'전체 평균 초기 만족도 high 비율: {overall_high_satisfaction_rate:.1f}%')
print(f'최소 표본 기준: 전체 {MIN_TOTAL_GAMES}개 이상, 유저 반응 {MIN_RESPONSE_GAMES}개 이상')

침묵 그룹: 6,737개
유저 반응 그룹: 4,904개
전체 평균 유저 반응 확보율: 42.1%
전체 평균 초기 만족도 high 비율: 71.7%
최소 표본 기준: 전체 30개 이상, 유저 반응 10개 이상


**해석:** 조합 단위 분석은 표본 수가 급격히 줄어들기 때문에, 최소 표본 기준을 넘는 셀만 해석하는 것이 중요하다.

## 2. 장르 × 가격대 조합별 2x2 매트릭스 계산

각 게임이 가진 장르를 다중 장르 방식으로 확장한 뒤, `장르 × 가격대` 조합별로 전체 게임 수, 유저 반응 게임 수, 초기 만족도 high 게임 수를 계산한다.

In [14]:
def explode_genre_price(df, include_satisfaction=False):
    frame = df.explode('genres').copy()
    frame = frame[frame['genres'].isin(TARGET_GENRES)].copy()
    frame = frame.dropna(subset=['price_range']).copy()
    frame['genre_price'] = frame['genres'] + ' × ' + frame['price_range'].astype(str)
    cols = ['appid', 'genres', 'price_range', 'genre_price']
    if include_satisfaction:
        cols.append('is_high_satisfaction')
    return frame[cols]

silence_gp = explode_genre_price(silence)
response_gp = explode_genre_price(response, include_satisfaction=True)
all_gp = pd.concat([silence_gp, response_gp[['appid', 'genres', 'price_range', 'genre_price']]], ignore_index=True)

combo_total = all_gp.groupby(['genres', 'price_range', 'genre_price'], observed=True)['appid'].count().rename('total_games')
combo_response = response_gp.groupby(['genres', 'price_range', 'genre_price'], observed=True)['appid'].count().rename('response_games')
combo_high = response_gp.groupby(['genres', 'price_range', 'genre_price'], observed=True)['is_high_satisfaction'].sum().rename('high_satisfaction_games')

combo_matrix = pd.concat([combo_total, combo_response, combo_high], axis=1).fillna(0).reset_index()
combo_matrix[['total_games', 'response_games', 'high_satisfaction_games']] = combo_matrix[['total_games', 'response_games', 'high_satisfaction_games']].astype(int)
combo_matrix = combo_matrix[(combo_matrix['total_games'] >= MIN_TOTAL_GAMES) & (combo_matrix['response_games'] >= MIN_RESPONSE_GAMES)].copy()
combo_matrix['유저 반응 확보율(%)'] = (combo_matrix['response_games'] / combo_matrix['total_games'] * 100).round(1)
combo_matrix['초기 만족도 high 비율(%)'] = (combo_matrix['high_satisfaction_games'] / combo_matrix['response_games'] * 100).round(1)
combo_matrix = combo_matrix.sort_values(['유저 반응 확보율(%)', '초기 만족도 high 비율(%)'], ascending=[False, False]).reset_index(drop=True)

total_exploded_games = len(all_gp)
total_exploded_response = len(response_gp)
total_exploded_high = int(response_gp['is_high_satisfaction'].sum())

print(f'해석 가능한 장르×가격대 조합 수: {len(combo_matrix):,}개')
display(combo_matrix.head(20))

해석 가능한 장르×가격대 조합 수: 36개


,genres,price_range,genre_price,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
0,Action,$20~30,Action × $20~30,44,32,18,72.7,56.2
1,Adventure,$15~20,Adventure × $15~20,193,121,80,62.7,66.1
2,Casual,$60+,Casual × $60+,30,18,7,60.0,38.9
3,RPG,$15~20,RPG × $15~20,89,53,32,59.6,60.4
4,Adventure,$10~15,Adventure × $10~15,417,248,195,59.5,78.6
5,Adventure,$20~30,Adventure × $20~30,59,35,20,59.3,57.1
6,Simulation,$10~15,Simulation × $10~15,167,98,61,58.7,62.2
7,Casual,$20~30,Casual × $20~30,52,30,17,57.7,56.7
8,Action,$10~15,Action × $10~15,377,217,162,57.6,74.7
9,Action,$15~20,Action × $15~20,155,88,52,56.8,59.1


**해석 가이드:** 같은 장르라도 가격대가 달라지면 위치가 달라질 수 있다. 즉 장르의 특성만이 아니라, 그 장르를 어떤 가격대로 포지셔닝했는지가 첫 반응과 만족도에 모두 영향을 준다고 볼 수 있다.

## 3. 장르 × 가격대 2x2 포지셔닝 시각화

각 조합을 하나의 점으로 두고, 전체 평균선 대비 어디에 놓이는지 본다. 점 크기는 해당 조합의 전체 게임 수다.

In [15]:
fig = px.scatter(
    combo_matrix,
    x='유저 반응 확보율(%)',
    y='초기 만족도 high 비율(%)',
    size='total_games',
    text='genre_price',
    color='genres',
    hover_data={
        'price_range': True,
        'total_games': True,
        'response_games': True,
        'high_satisfaction_games': True,
        '유저 반응 확보율(%)': True,
        '초기 만족도 high 비율(%)': True,
    },
    title='장르 × 가격대 조합별 유저 반응 확보율 × 초기 만족도 high 비율'
)
fig.add_vline(x=overall_response_capture_rate, line_dash='dash', line_color='gray')
fig.add_hline(y=overall_high_satisfaction_rate, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=1200, height=800)
fig.show()

**해석:**
- 우상단 조합은 실제 출시 전략 후보로 가장 매력적이다.
- 우하단 조합은 첫 클릭과 구매 전환은 일으키지만, 유저 기대치를 만족시키지 못했을 가능성이 크다.
- 좌상단 조합은 더 좋은 노출 전략을 붙이면 확장 여지가 있는 니치 조합일 수 있다.
- 좌하단 조합은 장르와 가격 포지셔닝 자체를 다시 검토할 필요가 있다.

## 4. 조합별 유의성 검정

조합별 차이가 우연인지 확인하기 위해 각 `장르 × 가격대` 조합을 **나머지 전체 조합**과 비교하는 2×2 Fisher exact test를 수행한다.

- `유저 반응 확보 검정`: 해당 조합의 반응 확보율이 나머지 전체보다 다른가
- `초기 만족도 검정`: 해당 조합의 초기 만족도 high 비율이 나머지 유저 반응 조합보다 다른가
- 다중비교 문제를 줄이기 위해 두 검정 각각에 대해 `FDR(BH)` 보정을 적용한다

주의: 장르는 다중 라벨로 explode했기 때문에, 이 검정은 **탐색적 유의성 점검**으로 해석하는 것이 적절하다.

In [16]:
def safe_fisher(table):
    odds_ratio, p_value = fisher_exact(table, alternative='two-sided')
    return odds_ratio, p_value

response_test_rows = []
satisfaction_test_rows = []

for row in combo_matrix.itertuples(index=False):
    # response capture: combo vs rest on response vs silence
    combo_response = int(row.response_games)
    combo_silence = int(row.total_games - row.response_games)
    rest_response = int(total_exploded_response - row.response_games)
    rest_silence = int((total_exploded_games - row.total_games) - rest_response)
    response_or, response_p = safe_fisher([[combo_response, combo_silence], [rest_response, rest_silence]])

    # satisfaction: combo vs rest within response group on high vs non-high
    combo_high = int(row.high_satisfaction_games)
    combo_non_high = int(row.response_games - row.high_satisfaction_games)
    rest_high = int(total_exploded_high - row.high_satisfaction_games)
    rest_non_high = int((total_exploded_response - row.response_games) - rest_high)
    satisfaction_or, satisfaction_p = safe_fisher([[combo_high, combo_non_high], [rest_high, rest_non_high]])

    response_test_rows.append({
        'genre_price': row.genre_price,
        'response_odds_ratio': response_or,
        'response_p': response_p,
    })
    satisfaction_test_rows.append({
        'genre_price': row.genre_price,
        'satisfaction_odds_ratio': satisfaction_or,
        'satisfaction_p': satisfaction_p,
    })

response_tests = pd.DataFrame(response_test_rows)
satisfaction_tests = pd.DataFrame(satisfaction_test_rows)
response_tests['response_fdr'] = fdrcorrection(response_tests['response_p'])[1]
satisfaction_tests['satisfaction_fdr'] = fdrcorrection(satisfaction_tests['satisfaction_p'])[1]

combo_matrix = combo_matrix.merge(response_tests, on='genre_price', how='left')
combo_matrix = combo_matrix.merge(satisfaction_tests, on='genre_price', how='left')
combo_matrix['response_sig'] = combo_matrix['response_fdr'] < 0.05
combo_matrix['satisfaction_sig'] = combo_matrix['satisfaction_fdr'] < 0.05

significance_view = combo_matrix[[
    'genre_price', '유저 반응 확보율(%)', '초기 만족도 high 비율(%)',
    'response_odds_ratio', 'response_fdr', 'response_sig',
    'satisfaction_odds_ratio', 'satisfaction_fdr', 'satisfaction_sig'
]].copy()

significance_view[['response_odds_ratio', 'response_fdr', 'satisfaction_odds_ratio', 'satisfaction_fdr']] = (
    significance_view[['response_odds_ratio', 'response_fdr', 'satisfaction_odds_ratio', 'satisfaction_fdr']].round(4)
)

display(significance_view.head(20))

print('반응 확보율 유의 조합 수:', int(combo_matrix['response_sig'].sum()))
print('초기 만족도 유의 조합 수:', int(combo_matrix['satisfaction_sig'].sum()))

,genre_price,유저 반응 확보율(%),초기 만족도 high 비율(%),response_odds_ratio,response_fdr,response_sig,satisfaction_odds_ratio,satisfaction_fdr,satisfaction_sig
0,Action × $20~30,72.7,56.2,3.5955,0.0003,True,0.5822,0.2121,False
1,Adventure × $15~20,62.7,66.1,2.2758,0.0000,True,0.8838,0.5864,False
2,Casual × $60+,60.0,38.9,2.0198,0.0860,False,0.2880,0.0492,True
3,RPG × $15~20,59.6,60.4,1.9857,0.0036,True,0.6899,0.2509,False
4,Adventure × $10~15,59.5,78.6,1.9981,0.0000,True,1.6879,0.0077,True
5,Adventure × $20~30,59.3,57.1,1.9652,0.0202,True,0.6037,0.2277,False
6,Simulation × $10~15,58.7,62.2,1.9194,0.0001,True,0.7457,0.2509,False
7,Casual × $20~30,57.7,56.7,1.8370,0.0478,True,0.5922,0.2439,False
8,Action × $10~15,57.6,74.7,1.8423,0.0000,True,1.3440,0.1356,False
9,Action × $15~20,56.8,59.1,1.7736,0.0010,True,0.6527,0.1356,False


반응 확보율 유의 조합 수: 26
초기 만족도 유의 조합 수: 7


## 5. 사분면별 조합 정리

해석을 쉽게 하기 위해 각 조합을 4개 사분면으로 나눈다.

In [17]:
def assign_quadrant(row):
    high_response = row['유저 반응 확보율(%)'] >= overall_response_capture_rate
    high_satisfaction = row['초기 만족도 high 비율(%)'] >= overall_high_satisfaction_rate
    if high_response and high_satisfaction:
        return '우상단: 반응 확보·만족도 모두 강함'
    if high_response and not high_satisfaction:
        return '우하단: 반응 확보는 강하지만 만족도 약함'
    if not high_response and high_satisfaction:
        return '좌상단: 반응 확보는 약하지만 만족도 강함'
    return '좌하단: 반응 확보·만족도 모두 약함'

combo_matrix['quadrant'] = combo_matrix.apply(assign_quadrant, axis=1)
quadrant_summary = combo_matrix.groupby('quadrant', observed=True).agg(
    combo_count=('genre_price', 'count'),
    median_response_rate=('유저 반응 확보율(%)', 'median'),
    median_high_satisfaction_rate=('초기 만족도 high 비율(%)', 'median'),
).reset_index()
display(quadrant_summary)

for quadrant in combo_matrix['quadrant'].unique():
    print(f'\n[{quadrant}]')
    display(combo_matrix.loc[combo_matrix['quadrant'] == quadrant, [
        'genre_price', 'total_games', 'response_games', 'high_satisfaction_games',
        '유저 반응 확보율(%)', '초기 만족도 high 비율(%)'
    ]].head(10))

,quadrant,combo_count,median_response_rate,median_high_satisfaction_rate
0,우상단: 반응 확보·만족도 모두 강함,9,54.80,74.7
1,우하단: 반응 확보는 강하지만 만족도 약함,18,54.15,58.1
2,좌상단: 반응 확보는 약하지만 만족도 강함,2,34.05,73.8
3,좌하단: 반응 확보·만족도 모두 약함,7,39.40,65.0



[우하단: 반응 확보는 강하지만 만족도 약함]


,genre_price,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
0,Action × $20~30,44,32,18,72.7,56.2
1,Adventure × $15~20,193,121,80,62.7,66.1
2,Casual × $60+,30,18,7,60.0,38.9
3,RPG × $15~20,89,53,32,59.6,60.4
5,Adventure × $20~30,59,35,20,59.3,57.1
6,Simulation × $10~15,167,98,61,58.7,62.2
7,Casual × $20~30,52,30,17,57.7,56.7
9,Action × $15~20,155,88,52,56.8,59.1
12,Simulation × $15~20,82,45,24,54.9,53.3
14,Casual × $15~20,116,62,33,53.4,53.2



[우상단: 반응 확보·만족도 모두 강함]


,genre_price,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
4,Adventure × $10~15,417,248,195,59.5,78.6
8,Action × $10~15,377,217,162,57.6,74.7
10,RPG × $10~15,182,103,82,56.6,79.6
11,Casual × $10~15,315,174,137,55.2,78.7
13,Strategy × $10~15,210,115,85,54.8,73.9
19,Action × $5~10,1351,665,486,49.2,73.1
22,Strategy × $5~10,599,289,209,48.2,72.3
23,Casual × $5~10,1236,574,420,46.4,73.2
24,Sports × $10~15,33,15,13,45.5,86.7



[좌하단: 반응 확보·만족도 모두 약함]


,genre_price,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
27,Adventure × ~$5,3206,1307,865,40.8,66.2
28,Sports × ~$5,243,99,64,40.7,64.6
29,Simulation × $20~30,37,15,5,40.5,33.3
30,Racing × $5~10,104,41,27,39.4,65.9
31,RPG × ~$5,1143,431,280,37.7,65.0
32,Racing × ~$5,289,105,64,36.3,61.0
33,Action × ~$5,3317,1189,850,35.8,71.5



[좌상단: 반응 확보는 약하지만 만족도 강함]


,genre_price,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
34,Casual × ~$5,4188,1466,1105,35.0,75.4
35,Strategy × ~$5,1424,471,340,33.1,72.2


**해석 가이드:** 이 표는 각 조합을 실제 의사결정 언어로 옮기는 역할을 한다. 우상단은 유지·확장 후보, 우하단은 기대치 관리 개선 후보, 좌상단은 노출 전략 개선 후보, 좌하단은 포지셔닝 재검토 후보로 읽으면 된다.

## 6. 종합 결론

이제 이 노트북에서는 단순 위치 비교를 넘어서, 다음 수준의 메시지를 더 엄밀하게 말할 수 있다.

1. **어떤 장르·가격대 조합은 유저 반응 확보율이 실제로 높고, 그 차이가 통계적으로도 유의하다.**
2. **어떤 조합은 반응 확보율은 높지만 초기 만족도 high 비율은 낮아, 기대치 관리 실패 가능성이 있다.**
3. **어떤 조합은 반응 확보율과 초기 만족도 high 비율이 모두 강하고, 두 축 중 하나 또는 둘 다에서 유의성이 확인된다.**
4. **FDR 보정 후에도 유의하지 않은 조합은 위치가 좋아 보여도 표본 변동 가능성을 함께 고려해야 한다.**

실무적으로는 우상단이면서 `response_sig`, `satisfaction_sig` 중 하나 이상이 `True`인 조합을 우선 전략 후보로 보고, 우하단이면서 만족도 검정이 유의한 조합은 기대치 조정 또는 품질 개선 우선 후보로 해석하면 된다.

다음 단계로는 태그 수집이 완료된 뒤 `장르 × 가격대 × 태그` 또는 `장르 × 태그` 조합으로 확장하면 실제 기획 방향 제안까지 더 구체화할 수 있다.